# Llama 3.1 8B activation extraction on Google Colab A100

This notebook reads the audited rendered prompts, applies the model's official chat template, and captures the residual-stream vector at the **last prompt token before answer generation**. It captures every transformer block by default and exports layer 24 separately in the repository-compatible `activations.npz` format.

Before running:

1. In Colab, select **Runtime → Change runtime type → A100 GPU**.
2. Accept access to [`meta-llama/Llama-3.1-8B-Instruct`](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct).
3. Add a Colab secret named `HF_TOKEN` containing a Hugging Face read token. Grant notebook access to the secret.
4. Put `prompts.grok43-medium_gpt54.full.audited.jsonl` at the configured Google Drive path, or use the upload fallback.

The extraction performs forward passes only; it does not generate or store model answers. Chunk checkpoints make the job resumable after a Colab disconnect.

In [ ]:
%pip -q install -U "transformers>=4.55,<5" "accelerate>=1.2" "safetensors>=0.4.5" "huggingface_hub>=0.27" "tqdm>=4.66"

In [ ]:
import gc
import hashlib
import json
import os
import platform
import shutil
import sys
import time
import zipfile
from collections import Counter
from getpass import getpass
from pathlib import Path

import numpy as np
import torch
import transformers
from huggingface_hub import HfApi, login
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GiB)')
print(f'Python {platform.python_version()} | torch {torch.__version__} | transformers {transformers.__version__}')
if 'A100' not in GPU_NAME.upper():
    print('WARNING: This notebook is tuned for an A100. Reduce BATCH_SIZE on a smaller GPU.')
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_grad_enabled(False)

## Storage and authentication

Google Drive is used for durable checkpoints. If the configured input file is absent, the cell offers a browser upload instead. The Hugging Face token is read from Colab Secrets and is never printed.

In [ ]:
from google.colab import drive, files, userdata

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/probe-experiment')

if USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass('Hugging Face read token: ')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face authentication configured.')

In [ ]:
# Experiment configuration
MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'
MODEL_REVISION = None  # Set to a full Hub commit hash to pin manually.
INPUT_FILENAME = 'prompts.grok43-medium_gpt54.full.audited.jsonl'
INPUT_JSONL = (DRIVE_PROJECT_DIR / 'outputs' / INPUT_FILENAME) if USE_GOOGLE_DRIVE else Path('/content') / INPUT_FILENAME
OUTPUT_ROOT = (DRIVE_PROJECT_DIR / 'activations') if USE_GOOGLE_DRIVE else Path('/content/activations')

BATCH_SIZE = 8
CHUNK_SIZE = 32
MAX_LENGTH = 2048
CAPTURE_LAYERS = 'all'  # Or a list such as [8, 16, 24, 31].
PRIMARY_LAYER = 24      # Exported as repository-compatible activations.npz.
STORAGE_DTYPE = np.float32

if not INPUT_JSONL.exists():
    print(f'Input not found at {INPUT_JSONL}. Upload the rendered prompt JSONL now.')
    uploaded = files.upload()
    if INPUT_FILENAME not in uploaded:
        if len(uploaded) != 1:
            raise FileNotFoundError(f'Upload exactly one file named {INPUT_FILENAME}.')
        uploaded_name = next(iter(uploaded))
        INPUT_JSONL = Path('/content') / uploaded_name
    else:
        INPUT_JSONL = Path('/content') / INPUT_FILENAME

print(f'Input: {INPUT_JSONL}')
print(f'Output root: {OUTPUT_ROOT}')

## Validate and fingerprint the rendered prompts

In [ ]:
def sha256_file(path: Path, chunk_size: int = 2**20) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_write_json(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    os.replace(temporary, path)

with INPUT_JSONL.open('r', encoding='utf-8') as stream:
    rows = [json.loads(line) for line in stream if line.strip()]

required = {
    'item_id', 'payload_block_id', 'topic_id', 'task_family',
    'intended_purpose', 'label', 'format', 'text',
    'rendered_text_sha256', 'split_group_id'
}
assert rows, 'Input JSONL is empty.'
for index, row in enumerate(rows):
    missing = required - row.keys()
    assert not missing, f'Row {index} is missing fields: {sorted(missing)}'
    actual_text_hash = hashlib.sha256(str(row['text']).encode('utf-8')).hexdigest()
    assert actual_text_hash == row['rendered_text_sha256'], f'Rendered text hash mismatch at row {index}'

item_ids = [str(row['item_id']) for row in rows]
assert len(item_ids) == len(set(item_ids)), 'Duplicate item_id values found.'
assert set(row['intended_purpose'] for row in rows) == {'evaluation', 'deployment'}
assert set(row['format'] for row in rows) == {'benchmark', 'casual'}

INPUT_SHA256 = sha256_file(INPUT_JSONL)
cells = Counter((row['intended_purpose'], row['format']) for row in rows)
print(f'Validated {len(rows)} prompts from {len(set(row["payload_block_id"] for row in rows))} payload blocks.')
print(f'Input SHA-256: {INPUT_SHA256}')
print('Cells:', dict(sorted(cells.items())))

## Load Llama 3.1 8B Instruct

The resolved Hub commit is recorded even when `MODEL_REVISION` is left unset. BF16 is used on the A100.

In [ ]:
hub_info = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN)
RESOLVED_REVISION = hub_info.sha
print(f'Resolved model revision: {RESOLVED_REVISION}')

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=RESOLVED_REVISION, token=HF_TOKEN
)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=RESOLVED_REVISION,
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
    attn_implementation='sdpa',
)
model.eval()
assert hasattr(model, 'model') and hasattr(model.model, 'layers'), 'Expected a Hugging Face Llama causal-LM architecture.'
assert all(str(device) not in {'cpu', 'disk'} for device in model.hf_device_map.values()), model.hf_device_map

N_LAYERS = int(model.config.num_hidden_layers)
HIDDEN_SIZE = int(model.config.hidden_size)
LAYERS = list(range(N_LAYERS)) if CAPTURE_LAYERS == 'all' else [int(layer) for layer in CAPTURE_LAYERS]
assert LAYERS == sorted(set(LAYERS)), 'CAPTURE_LAYERS must be sorted and unique.'
assert all(0 <= layer < N_LAYERS for layer in LAYERS)
assert PRIMARY_LAYER in LAYERS
INPUT_DEVICE = model.get_input_embeddings().weight.device
print(f'{N_LAYERS} transformer blocks, hidden size {HIDDEN_SIZE}, input device {INPUT_DEVICE}')
print(f'Capturing {len(LAYERS)} layers: {LAYERS}')

## Apply the chat template and check token lengths

Hugging Face recommends using each chat model's own template. Because the template is first rendered as text and then batch-tokenized, `add_special_tokens=False` prevents duplicated control tokens. The notebook fails rather than silently truncating a prompt.

In [ ]:
formatted_prompts = [
    tokenizer.apply_chat_template(
        [{'role': 'user', 'content': str(row['text'])}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for row in rows
]

unbatched_ids = tokenizer(
    formatted_prompts,
    add_special_tokens=False,
    padding=False,
    truncation=False,
)['input_ids']
token_lengths = np.asarray([len(ids) for ids in unbatched_ids], dtype=np.int32)
assert int(token_lengths.max()) <= MAX_LENGTH, (
    f'Longest prompt has {token_lengths.max()} tokens, exceeding MAX_LENGTH={MAX_LENGTH}. '
    'Raise MAX_LENGTH intentionally; do not silently truncate.'
)
print(
    f'Token lengths: min={token_lengths.min()}, median={int(np.median(token_lengths))}, '
    f'max={token_lengths.max()}'
)
print('Final tokens of first formatted prompt:')
print(repr(formatted_prompts[0][-240:]))

## Configure resumable extraction

A forward hook stores only the last-token vector from each selected block, avoiding the large memory overhead of returning every sequence position from every layer. Checkpoints are keyed by the input hash, exact model commit, layer set, and tokenization settings.

In [ ]:
plan_core = {
    'schema_version': 1,
    'model': MODEL_ID,
    'model_revision': RESOLVED_REVISION,
    'input_sha256': INPUT_SHA256,
    'n_items': len(rows),
    'layers': LAYERS,
    'primary_layer': PRIMARY_LAYER,
    'position': 'last_prompt_token_after_chat_template_before_generation',
    'layer_semantics': 'zero_based_transformer_block_output_before_final_model_norm',
    'chat_template': 'tokenizer.apply_chat_template(user_message, add_generation_prompt=True)',
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'chunk_size': CHUNK_SIZE,
    'attention_implementation': 'sdpa',
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'storage_dtype': np.dtype(STORAGE_DTYPE).name,
}
plan_fingerprint = hashlib.sha256(
    json.dumps(plan_core, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()[:16]
RUN_DIR = OUTPUT_ROOT / f'llama31_8b_{plan_fingerprint}'
CHUNK_DIR = RUN_DIR / 'chunks'
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
PLAN_PATH = RUN_DIR / 'extraction_plan.json'

if PLAN_PATH.exists():
    existing_plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
    assert existing_plan == plan_core, 'Existing checkpoint plan does not match this run.'
else:
    atomic_write_json(PLAN_PATH, plan_core)

print(f'Checkpoint directory: {RUN_DIR}')

In [ ]:
batch_capture = {}

def make_last_token_hook(layer_index):
    def hook(_module, _inputs, output):
        hidden = output[0] if isinstance(output, (tuple, list)) else output
        batch_capture[layer_index] = hidden[:, -1, :].detach()
    return hook

handles = [
    model.model.layers[layer].register_forward_hook(make_last_token_hook(layer))
    for layer in LAYERS
]

def extract_batch(prompt_texts):
    batch_capture.clear()
    encoded = tokenizer(
        prompt_texts,
        add_special_tokens=False,
        return_tensors='pt',
        padding=True,
        truncation=False,
    )
    assert bool(torch.all(encoded['attention_mask'][:, -1] == 1)), 'Last position must be a real prompt token.'
    encoded = {key: value.to(INPUT_DEVICE) for key, value in encoded.items()}
    with torch.inference_mode():
        base_output = model.model(**encoded, use_cache=False, return_dict=True)
    missing = set(LAYERS) - batch_capture.keys()
    assert not missing, f'Hooks did not fire for layers: {sorted(missing)}'
    stacked = torch.stack([batch_capture[layer] for layer in LAYERS], dim=1)
    result = stacked.float().cpu().numpy().astype(STORAGE_DTYPE, copy=False)
    lengths = encoded['attention_mask'].sum(dim=1).to(torch.int32).cpu().numpy()
    del base_output, stacked, encoded
    batch_capture.clear()
    return result, lengths

print('Hooks registered. Expected batch output shape:', (BATCH_SIZE, len(LAYERS), HIDDEN_SIZE))

## Run extraction

Completed chunks are validated and skipped on rerun. A chunk is written atomically only after all of its batches finish.

In [ ]:
started = time.time()
try:
    for chunk_start in tqdm(range(0, len(rows), CHUNK_SIZE), desc='Activation chunks'):
        chunk_end = min(chunk_start + CHUNK_SIZE, len(rows))
        chunk_path = CHUNK_DIR / f'chunk_{chunk_start:05d}_{chunk_end - 1:05d}.npz'
        expected_ids = np.asarray(item_ids[chunk_start:chunk_end], dtype=str)

        if chunk_path.exists():
            with np.load(chunk_path, allow_pickle=False) as saved:
                valid = (
                    saved['X'].shape == (chunk_end - chunk_start, len(LAYERS), HIDDEN_SIZE)
                    and np.array_equal(saved['item_ids'], expected_ids)
                    and np.array_equal(saved['layers'], np.asarray(LAYERS, dtype=np.int16))
                )
            if valid:
                continue
            raise RuntimeError(f'Invalid existing checkpoint: {chunk_path}')

        vectors = []
        lengths = []
        for batch_start in range(chunk_start, chunk_end, BATCH_SIZE):
            batch_end = min(batch_start + BATCH_SIZE, chunk_end)
            batch_vectors, batch_lengths = extract_batch(formatted_prompts[batch_start:batch_end])
            vectors.append(batch_vectors)
            lengths.append(batch_lengths)

        chunk_X = np.concatenate(vectors, axis=0)
        chunk_lengths = np.concatenate(lengths, axis=0)
        assert chunk_X.shape == (chunk_end - chunk_start, len(LAYERS), HIDDEN_SIZE)
        temporary = chunk_path.with_suffix('.tmp.npz')
        np.savez_compressed(
            temporary,
            X=chunk_X,
            item_ids=expected_ids,
            layers=np.asarray(LAYERS, dtype=np.int16),
            token_lengths=chunk_lengths,
        )
        os.replace(temporary, chunk_path)
        del chunk_X, chunk_lengths, vectors, lengths
        gc.collect()
        torch.cuda.empty_cache()
finally:
    for handle in handles:
        handle.remove()

print(f'Extraction/checkpoint pass finished in {(time.time() - started) / 60:.1f} minutes.')

## Consolidate and verify outputs

`activations_all_layers.npz` contains `X` with shape `[items, layers, hidden_size]`. `activations.npz` contains the layer-24 matrix `X` with shape `[items, hidden_size]`, matching the repository's analysis loader. `meta.jsonl` preserves the exact rendered-item order.

In [ ]:
chunk_paths = sorted(CHUNK_DIR.glob('chunk_*.npz'))
expected_chunk_count = (len(rows) + CHUNK_SIZE - 1) // CHUNK_SIZE
assert len(chunk_paths) == expected_chunk_count, (len(chunk_paths), expected_chunk_count)

all_X_parts = []
all_id_parts = []
all_length_parts = []
for chunk_path in chunk_paths:
    with np.load(chunk_path, allow_pickle=False) as saved:
        all_X_parts.append(saved['X'])
        all_id_parts.append(saved['item_ids'])
        all_length_parts.append(saved['token_lengths'])

X_all = np.concatenate(all_X_parts, axis=0)
saved_item_ids = np.concatenate(all_id_parts, axis=0)
saved_token_lengths = np.concatenate(all_length_parts, axis=0)
assert X_all.shape == (len(rows), len(LAYERS), HIDDEN_SIZE)
assert np.array_equal(saved_item_ids, np.asarray(item_ids, dtype=str))
assert np.array_equal(saved_token_lengths, token_lengths)
assert np.isfinite(X_all).all(), 'Non-finite activations found.'

ALL_LAYERS_PATH = RUN_DIR / 'activations_all_layers.npz'
PRIMARY_PATH = RUN_DIR / 'activations.npz'
META_PATH = RUN_DIR / 'meta.jsonl'
CONFIG_PATH = RUN_DIR / 'config.json'

np.savez_compressed(
    ALL_LAYERS_PATH,
    X=X_all,
    item_ids=saved_item_ids,
    layers=np.asarray(LAYERS, dtype=np.int16),
    token_lengths=saved_token_lengths,
)
primary_index = LAYERS.index(PRIMARY_LAYER)
np.savez_compressed(PRIMARY_PATH, X=X_all[:, primary_index, :], item_ids=saved_item_ids)
shutil.copyfile(INPUT_JSONL, META_PATH)

config = {
    **plan_core,
    'run_fingerprint': plan_fingerprint,
    'n_transformer_layers': N_LAYERS,
    'hidden_size': HIDDEN_SIZE,
    'batch_size': BATCH_SIZE,
    'chunk_size': CHUNK_SIZE,
    'dtype_inference': 'bfloat16',
    'dtype_stored': np.dtype(STORAGE_DTYPE).name,
    'gpu': GPU_NAME,
    'gpu_memory_gib': round(GPU_MEMORY_GB, 2),
    'python_version': platform.python_version(),
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'input_path': str(INPUT_JSONL),
    'outputs': {
        'all_layers': {'path': str(ALL_LAYERS_PATH), 'sha256': sha256_file(ALL_LAYERS_PATH)},
        'primary_layer': {'path': str(PRIMARY_PATH), 'sha256': sha256_file(PRIMARY_PATH)},
        'metadata': {'path': str(META_PATH), 'sha256': sha256_file(META_PATH)},
    },
}
atomic_write_json(CONFIG_PATH, config)

print(f'All-layer shape: {X_all.shape}')
print(f'Primary-layer shape: {X_all[:, primary_index, :].shape}')
print(f'Output directory: {RUN_DIR}')
print(f'All-layer file: {ALL_LAYERS_PATH} ({ALL_LAYERS_PATH.stat().st_size / 2**20:.1f} MiB)')
print(f'Layer-{PRIMARY_LAYER} file: {PRIMARY_PATH} ({PRIMARY_PATH.stat().st_size / 2**20:.1f} MiB)')

In [ ]:
# Final alignment checks and a compact activation diagnostic.
with np.load(PRIMARY_PATH, allow_pickle=False) as primary:
    assert primary['X'].shape == (len(rows), HIDDEN_SIZE)
    assert np.array_equal(primary['item_ids'], np.asarray(item_ids, dtype=str))

layer_norm_means = np.linalg.norm(X_all, axis=2).mean(axis=0)
for layer, mean_norm in zip(LAYERS, layer_norm_means):
    print(f'layer {layer:02d}: mean L2 norm = {mean_norm:.4f}')

print('\nExtraction verified: item order, shapes, token lengths, hashes, and finite values all pass.')

## Optional download

When Drive is enabled, the durable files are already stored there. The cell below creates a ZIP containing only the consolidated outputs—not the redundant checkpoints—and downloads it through the browser.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    archive_path = Path('/content') / f'{RUN_DIR.name}.zip'
    with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in (ALL_LAYERS_PATH, PRIMARY_PATH, META_PATH, CONFIG_PATH, PLAN_PATH):
            archive.write(path, arcname=path.name)
    print(f'Downloading {archive_path} ({archive_path.stat().st_size / 2**20:.1f} MiB)')
    files.download(str(archive_path))
else:
    print(f'Results are available at {RUN_DIR}')